In [0]:
from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"

df_silver = spark.table(silver_table)

df_gold = (
    df_silver
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.sum(
            F.when(F.col("email") == "unknown", 1).otherwise(0)
        ).alias("customers_missing_email")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

display(df_gold)

city,total_customers,customers_missing_email,gold_load_timestamp
Pune,2,0,2026-10-14T06:40:49.018Z
Mumbai,1,0,2026-10-14T06:40:49.018Z
Nashik,1,1,2026-10-14T06:40:49.018Z
Nagpur,1,0,2026-10-14T06:40:49.018Z


In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table)

print("Gold customer analytics table created successfully")

Gold customer analytics table created successfully


In [0]:
%sql
SELECT *
FROM myadlasdevcalaog.gold.customer_city_summary
ORDER BY total_customers DESC;

city,total_customers,customers_missing_email,gold_load_timestamp
Pune,2,0,2026-10-14T06:42:24.061Z
Nashik,1,1,2026-10-14T06:42:24.061Z
Mumbai,1,0,2026-10-14T06:42:24.061Z
Nagpur,1,0,2026-10-14T06:42:24.061Z


In [0]:
from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"

df_silver = spark.table(silver_table)

df_gold = (
    df_silver
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.sum(
            F.when(
                F.col("email").isNull() |
                (F.trim(F.col("email")) == "") |
                (F.col("email") == "unknown"),
                1
            ).otherwise(0)
        ).alias("customers_missing_email")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_table)

display(df_gold.orderBy("city"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5491224525003479>, line 25
      6 df_silver = spark.table(silver_table)
      8 df_gold = (
      9     df_silver
     10     .groupBy("city")
   (...)     22     .withColumn("gold_load_timestamp", F.current_timestamp())
     23 )
---> 25 df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_table)
     27 display(df_gold.orderBy("city"))

File <command-5491224525003479>, line 9
      4 gold_table = "myadlasdevcalaog.gold.customer_city_summary"
      6 df_silver = spark.table(silver_table)
      8 df_gold = (
----> 9     df_silver
     10     .groupBy("city")
     11     .agg(
     12         F.countDistinct("customer_id").alias("total_customers"),
     13         F.sum(
     14             F.when(
     15                 F.col("email").isNull() |
     16                 (F.trim(F.col("email")) == "") |
     17

In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"

df_silver = spark.table(silver_table)

df_gold = (
    df_silver
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.sum(
            F.when(
                F.col("email").isNull()
                | (F.trim(F.col("email")) == "")
                | (F.col("email") == "unknown"),
                1
            ).otherwise(0)
        ).alias("customers_missing_email")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table)

display(df_gold.orderBy("city"))


city,total_customers,customers_missing_email,gold_load_timestamp
Bengaluru,1,0,2026-10-16T09:54:38.800Z
Mumbai,1,0,2026-10-16T09:54:38.800Z
Nagpur,1,0,2026-10-16T09:54:38.800Z
Nashik,1,1,2026-10-16T09:54:38.800Z
Pune,2,0,2026-10-16T09:54:38.800Z


In [0]:
%sql
SELECT *
FROM myadlasdevcalaog.gold.customer_city_summary
ORDER BY city;

city,total_customers,customers_missing_email,gold_load_timestamp
Bengaluru,1,0,2026-10-16T09:54:36.370Z
Mumbai,1,0,2026-10-16T09:54:36.370Z
Nagpur,1,0,2026-10-16T09:54:36.370Z
Nashik,1,1,2026-10-16T09:54:36.370Z
Pune,2,0,2026-10-16T09:54:36.370Z


In [0]:
%sql
SELECT SUM(total_customers) AS total_customers
FROM myadlasdevcalaog.gold.customer_city_summary;

total_customers
6


In [0]:
%sql
SELECT city, COUNT(*) AS record_count
FROM myadlasdevcalaog.gold.customer_city_summary
GROUP BY city
HAVING COUNT(*) > 1;

city,record_count


In [0]:
%sql
SELECT SUM(customers_missing_email) AS missing_email_count
FROM myadlasdevcalaog.gold.customer_city_summary;

missing_email_count
1
